# AI-Powered Lung Disease Detection
## Multi-Class Preprocessing, EDA, Bias Analysis & YOLOX Training

This notebook implements a **state-of-the-art preprocessing pipeline** for detecting **lung tumors** and **tuberculosis (TB)** from chest X-ray images using the YOLOX anchor-free object detection framework.

### Data Sources
| Dataset | Disease | Annotation Type | Format |
|---------|---------|----------------|--------|
| Lung Tumor Dataset | Cancer/Tumor | Segmentation Masks → Bounding Boxes | PNG masks |
| TBX11K | Tuberculosis | COCO-style Bounding Boxes | JSON annotations |

### Pipeline Overview
1. **Multi-Source Data Loading** — Unified annotation parsing from heterogeneous formats
2. **Raw Data Quality Audit** — Image quality profiling, resolution analysis
3. **Data Preparation & Integrity** — Corrupt detection, duplicates, format standardization
4. **Exploratory Data Analysis** — Intensity distributions, spatial heatmaps, bbox statistics
5. **Bias Analysis & Detection** — Cross-dataset domain shift, spatial/size bias, class imbalance
6. **Bias Mitigation** — Weighted sampling, size-aware augmentation
7. **Image Preprocessing** — CLAHE enhancement, bilateral denoising
8. **Stratified Splitting** — Leakage-free train/val/test with balanced representation
9. **Advanced Augmentation** — Medical-imaging-specific augmentation pipeline
10. **Data Integrity Validation** — Final verification before training

## 1. Environment Setup

In [ ]:
%%capture
# Install PyTorch with CUDA support
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# Install core visualization and data handling
!pip install matplotlib pandas pillow torchtnt==0.2.0 tqdm opencv-python seaborn
# Install data format support
!pip install tabulate pyarrow fastparquet
# Install colormap utilities
!pip install distinctipy
# Install COCO tools for TBX11K annotation parsing
!pip install pycocotools
# Install perceptual hashing for duplicate detection
!pip install imagehash
# Install statistical and ML utilities
!pip install scipy scikit-learn
# Install YOLOX and custom utility packages
!pip install cjm_pandas_utils cjm_psl_utils cjm_pil_utils cjm_pytorch_utils cjm_yolox_pytorch cjm_torchvision_tfms

In [ ]:
# Python Standard Library
import datetime
from functools import partial
from glob import glob
import hashlib, json, math, multiprocessing, os, random, warnings
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')

# Utility packages
from cjm_psl_utils.core import download_file, file_extract
from cjm_pil_utils.core import resize_img, get_img_files, stack_imgs
from cjm_pytorch_utils.core import tensor_to_pil, get_torch_device, set_seed, denorm_img_tensor
from cjm_pandas_utils.core import markdown_to_pandas, convert_to_numeric, convert_to_string
from cjm_torchvision_tfms.core import ResizeMax, PadSquare, CustomRandomIoUCrop, CustomRandomAugment

# YOLOX
from cjm_yolox_pytorch.model import build_model, MODEL_CFGS, NORM_STATS
from cjm_yolox_pytorch.utils import generate_output_grids
from cjm_yolox_pytorch.loss import YOLOXLoss
from cjm_yolox_pytorch.inference import YOLOXInferenceWrapper

# Visualization
from distinctipy import distinctipy
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

# Scientific computing
import numpy as np
import cv2
from scipy import stats
from scipy.stats import chi2_contingency
import imagehash

# Data handling
import pandas as pd
pd.set_option('max_colwidth', None, 'display.max_rows', None, 'display.max_columns', None)

# PIL
from PIL import Image, ImageFilter

# PyTorch
import torch
from torch.amp import autocast
from torch.cuda.amp import GradScaler
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchtnt.utils import get_module_summary

# Torchvision
import torchvision
torchvision.disable_beta_transforms_warning()
from torchvision.tv_tensors import BoundingBoxes
from torchvision.utils import draw_bounding_boxes
import torchvision.transforms.v2 as transforms
from torchvision.transforms.v2 import functional as TF

# Progress & ML
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

print("All dependencies loaded successfully \u2713")

## 2. Configuration & Dataset Paths
Setting random seeds for reproducibility and defining paths for both data sources.

In [ ]:
seed = 42
set_seed(seed)
device = get_torch_device()
dtype = torch.float32

train_sz = 512
bs = 16
epochs = 50
lr = 1e-3

project_dir = Path("/kaggle/working/yolox_lung_disease")
project_dir.mkdir(parents=True, exist_ok=True)

# Dataset 1: Lung Tumor
tumor_image_dir = "/kaggle/input/pidata-new-names/Dataset/Images"
tumor_mask_dir  = "/kaggle/input/pidata-new-names/Dataset/Annotations"

# Dataset 2: TBX11K
tbx_root      = "/kaggle/input/tbx11k/TBX11K"
tbx_anno_file = os.path.join(tbx_root, "annotations", "json", "TBX11K_trainval_only_tb.json")
tbx_image_dir = os.path.join(tbx_root, "imgs")

CLASS_NAMES = ['tumor', 'tuberculosis']
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

print(f"Device: {device}")
print(f"Resolution: {train_sz}x{train_sz}")
print(f"Classes: {CLASS_NAMES}")

## 3. Multi-Source Data Loading

Our two datasets use entirely different annotation formats:
- **Tumor**: Binary segmentation masks (PNG) \u2192 contour analysis \u2192 bounding boxes
- **TBX11K**: COCO-format JSON with pre-annotated bounding boxes

We unify both into a single DataFrame.

### 3.1 Tumor Dataset \u2014 Mask-to-BBox with Morphological Cleanup

In [ ]:
def mask_to_bbox_advanced(mask_path, img_path, min_area=50):
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    img = cv2.imread(str(img_path))
    if mask is None or img is None: return []
    img_h, img_w = img.shape[:2]
    _, binary = cv2.threshold(mask, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel, iterations=1)
    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for c in contours:
        a = cv2.contourArea(c)
        if a > min_area:
            x,y,w,h = cv2.boundingRect(c)
            ba = w*h
            bboxes.append({'xmin':x,'ymin':y,'xmax':x+w,'ymax':y+h,'width':w,'height':h,
                'area':ba,'relative_area_pct':round(ba/(img_w*img_h)*100,2),
                'aspect_ratio':round(w/max(h,1),3),
                'center_x':round((x+w/2)/img_w,3),'center_y':round((y+h/2)/img_h,3),
                'contour_area':a,'mask_fill_ratio':round(a/max(ba,1),3)})
    return bboxes

tumor_data = []
if os.path.exists(tumor_image_dir) and os.path.exists(tumor_mask_dir):
    img_f = {os.path.splitext(f)[0]:f for f in os.listdir(tumor_image_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))}
    msk_f = {os.path.splitext(f)[0]:f for f in os.listdir(tumor_mask_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))}
    common = sorted(set(img_f) & set(msk_f))
    print(f"Tumor: {len(img_f)} images, {len(msk_f)} masks, {len(common)} matched")
    no_bb = 0
    for k in tqdm(common, desc="Extracting tumor bboxes"):
        ip = os.path.join(tumor_image_dir, img_f[k])
        mp = os.path.join(tumor_mask_dir, msk_f[k])
        bb = mask_to_bbox_advanced(mp, ip)
        if not bb: no_bb += 1; continue
        for b in bb:
            tumor_data.append({'image':img_f[k],'image_path':ip,'label':'tumor','source_dataset':'lung_tumor',**b})
    print(f"  \u2192 {len(tumor_data)} bboxes ({no_bb} images skipped)")
else:
    print("\u26a0 Tumor dataset not found.")

### 3.2 TBX11K Dataset \u2014 COCO JSON Parsing

In [ ]:
tb_data = []
if os.path.exists(tbx_anno_file):
    with open(tbx_anno_file, 'r') as f:
        coco = json.load(f)
    id2img = {img['id']:img for img in coco['images']}
    id2cat = {cat['id']:cat['name'] for cat in coco['categories']}
    print(f"TBX11K: {len(coco['images'])} images, {len(coco['annotations'])} annotations")
    print(f"  Categories: {id2cat}")
    tb_ids = {cid for cid,n in id2cat.items() if any(k in n.lower() for k in ['tb','tuberculosis','active','latent'])}
    sk = 0
    for ann in tqdm(coco['annotations'], desc="Parsing TBX11K"):
        if ann['category_id'] not in tb_ids: sk+=1; continue
        ii = id2img.get(ann['image_id'])
        if ii is None: continue
        x,y,w,h = ann['bbox']
        iw,ih,fn = ii['width'],ii['height'],ii['file_name']
        ip = None
        for sd in ['','tb','sick','health']:
            c = os.path.join(tbx_image_dir,sd,fn) if sd else os.path.join(tbx_image_dir,fn)
            if os.path.exists(c): ip=c; break
        if ip is None:
            for root,_,files in os.walk(tbx_image_dir):
                if fn in files: ip=os.path.join(root,fn); break
        if ip is None: continue
        ba = int(w*h)
        tb_data.append({'image':fn,'image_path':ip,'label':'tuberculosis','source_dataset':'TBX11K',
            'xmin':int(x),'ymin':int(y),'xmax':int(x+w),'ymax':int(y+h),
            'width':int(w),'height':int(h),'area':ba,
            'relative_area_pct':round(ba/(iw*ih)*100,2),
            'aspect_ratio':round(w/max(h,1),3),
            'center_x':round((x+w/2)/iw,3),'center_y':round((y+h/2)/ih,3),
            'contour_area':ba,'mask_fill_ratio':1.0})
    print(f"  \u2192 {len(tb_data)} TB bboxes ({sk} non-TB skipped)")
else:
    print("\u26a0 TBX11K not found.")

### 3.3 Unified Dataset

In [ ]:
df = pd.DataFrame(tumor_data + tb_data)
if len(df) > 0:
    print("=" * 60)
    print("UNIFIED DATASET SUMMARY")
    print("=" * 60)
    print(f"Total annotations: {len(df)}, Unique images: {df['image'].nunique()}")
    for l in CLASS_NAMES:
        s = df[df['label']==l]
        print(f"  {l:15s} \u2192 {len(s):5d} bboxes, {s['image'].nunique():5d} images")
    display(df.head())

## 4. Raw Data Quality Audit
Thorough audit of image quality metrics across both data sources.

### 4.1 Image Quality Profiling

In [ ]:
def compute_qm(path):
    img = cv2.imread(path)
    if img is None: return None
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h,w = g.shape
    br = np.mean(g); co = np.std(g)
    sh = cv2.Laplacian(g, cv2.CV_64F).var()
    return {'width':w,'height':h,'aspect_ratio':round(w/max(h,1),3),
            'brightness':round(br,2),'contrast':round(co,2),'sharpness':round(sh,2),'snr':round(br/max(co,1e-6),3)}

if len(df) > 0:
    uq = df.drop_duplicates('image')[['image','image_path','source_dataset']].reset_index(drop=True)
    qr, fl = [], []
    for _,r in tqdm(uq.iterrows(), total=len(uq), desc="Quality profiling"):
        m = compute_qm(r['image_path'])
        if m: qr.append({'image':r['image'],'source_dataset':r['source_dataset'],**m})
        else: fl.append(r['image'])
    quality_df = pd.DataFrame(qr)
    if fl: print(f"\u26a0 {len(fl)} failed")
    print(f"Profiled {len(quality_df)} images")
    display(quality_df.describe().round(2))

### 4.2 Cross-Dataset Quality Comparison

In [ ]:
if len(df) > 0 and len(quality_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Image Quality \u2014 Cross-Dataset Comparison', fontsize=16, fontweight='bold')
    mets = ['brightness','contrast','sharpness','snr','width','height']
    tits = ['Brightness','Contrast','Sharpness','SNR','Width (px)','Height (px)']
    cols = {'lung_tumor':'#e74c3c','TBX11K':'#3498db'}
    for i,(m,t) in enumerate(zip(mets,tits)):
        ax = axes[i//3][i%3]
        for src in quality_df['source_dataset'].unique():
            ax.hist(quality_df[quality_df['source_dataset']==src][m], bins=30, alpha=0.6, label=src, color=cols.get(src,'#95a5a6'), edgecolor='white')
        ax.set_title(t, fontweight='bold'); ax.legend()
    plt.tight_layout(); plt.show()

    print("\nDOMAIN SHIFT ANALYSIS (Mann-Whitney U)")
    print("=" * 60)
    srcs = quality_df['source_dataset'].unique()
    if len(srcs)==2:
        for m in ['brightness','contrast','sharpness','snr']:
            g1 = quality_df[quality_df['source_dataset']==srcs[0]][m].dropna()
            g2 = quality_df[quality_df['source_dataset']==srcs[1]][m].dropna()
            if len(g1)>0 and len(g2)>0:
                st,p = stats.mannwhitneyu(g1,g2,alternative='two-sided')
                sig = "\u26a0 SIGNIFICANT" if p<0.05 else "\u2713 Not significant"
                print(f"  {m:12s} | U={st:10.0f} | p={p:.4e} | {sig}")

## 5. Data Preparation & Integrity

### 5.1 Corrupt Image Detection & Format Check

In [ ]:
if len(df) > 0:
    corrupt, gs_imgs = [], []
    uq = df.drop_duplicates('image')[['image','image_path','source_dataset']].reset_index(drop=True)
    for _,r in tqdm(uq.iterrows(), total=len(uq), desc="Integrity check"):
        try:
            img = Image.open(r['image_path']); img.verify()
            img = Image.open(r['image_path'])
            if img.mode == 'L': gs_imgs.append(r['image'])
        except Exception as e:
            corrupt.append(r['image'])
    print(f"\u2713 Checked: {len(uq)}, Corrupt: {len(corrupt)}, Grayscale: {len(gs_imgs)}")
    if corrupt:
        df = df[~df['image'].isin(corrupt)].reset_index(drop=True)
        print(f"  Removed {len(corrupt)} corrupt images")

### 5.2 Duplicate Detection (Perceptual Hashing)

In [ ]:
if len(df) > 0:
    hrecs = []
    uq = df.drop_duplicates('image')[['image','image_path','source_dataset']].reset_index(drop=True)
    for _,r in tqdm(uq.iterrows(), total=len(uq), desc="pHash"):
        try:
            img = Image.open(r['image_path']).convert('RGB')
            hrecs.append({'image':r['image'],'source_dataset':r['source_dataset'],'phash':str(imagehash.phash(img,hash_size=16))})
        except: pass
    hash_df = pd.DataFrame(hrecs)
    dups = hash_df['phash'].value_counts()
    dups = dups[dups>1]
    cd = 0
    if len(dups)>0:
        print(f"\u26a0 {len(dups)} duplicate groups:")
        for h in list(dups.index)[:5]:
            g = hash_df[hash_df['phash']==h]
            ic = len(g['source_dataset'].unique())>1
            if ic: cd+=1
            print(f"  {'CROSS-DATASET' if ic else 'within'}: {list(g['image'].values)}")
    else:
        print("\u2713 No near-duplicates")
    print(f"Unique hashes: {hash_df['phash'].nunique()}/{len(hash_df)}")

### 5.3 Mask Integrity Validation

In [ ]:
if len(df) > 0 and os.path.exists(tumor_mask_dir):
    issues = []
    ms = [f for f in os.listdir(tumor_mask_dir) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    for mn in tqdm(ms, desc="Mask check"):
        m = cv2.imread(os.path.join(tumor_mask_dir,mn), cv2.IMREAD_GRAYSCALE)
        if m is None: issues.append((mn,'load failed')); continue
        nbin = [v for v in np.unique(m) if v not in [0,255]]
        if len(nbin)>2: issues.append((mn,f'non-binary'))
        cov = np.sum(m>127)/m.size*100
        if cov>50: issues.append((mn,f'high coverage {cov:.1f}%'))
    print(f"{'\u26a0 '+str(len(issues))+' issues' if issues else '\u2713 All '+str(len(ms))+' masks OK'}")

## 6. In-Depth Exploratory Data Analysis (EDA)

### 6.1 Per-Channel Pixel Intensity

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    fig.suptitle('Per-Channel Pixel Intensity by Class', fontsize=14, fontweight='bold')
    chn = ['Red','Green','Blue']
    for label in CLASS_NAMES:
        paths = df[df['label']==label].drop_duplicates('image')['image_path'].tolist()
        sp = random.sample(paths, min(100, len(paths)))
        ch = [[],[],[]]
        for p in sp:
            img = cv2.imread(p)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                for c in range(3): ch[c].extend(img[:,:,c].flatten()[::50])
        for c in range(3):
            axes[c].hist(ch[c], bins=50, alpha=0.5, label=label, density=True, edgecolor='none')
            axes[c].set_title(chn[c], fontweight='bold'); axes[c].legend()
    plt.tight_layout(); plt.show()

### 6.2 Spatial Heatmap of Disease Locations

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(1, len(CLASS_NAMES)+1, figsize=(7*(len(CLASS_NAMES)+1), 6))
    fig.suptitle('Spatial Density Heatmap', fontsize=14, fontweight='bold')
    cs = 100
    for idx, label in enumerate(CLASS_NAMES):
        hm = np.zeros((cs,cs))
        sub = df[df['label']==label]
        for _, r in sub.iterrows():
            cx,cy = np.clip(int(r['center_x']*(cs-1)),0,cs-1), np.clip(int(r['center_y']*(cs-1)),0,cs-1)
            for dx in range(-3,4):
                for dy in range(-3,4):
                    nx,ny = cx+dx,cy+dy
                    if 0<=nx<cs and 0<=ny<cs: hm[ny,nx] += np.exp(-(dx**2+dy**2)/4.0)
        im = axes[idx].imshow(hm, cmap='YlOrRd', interpolation='gaussian')
        axes[idx].set_title(f'{label.upper()} ({len(sub)})', fontweight='bold')
        plt.colorbar(im, ax=axes[idx], fraction=0.046)
    hm_all = np.zeros((cs,cs))
    for _, r in df.iterrows():
        cx,cy = np.clip(int(r['center_x']*(cs-1)),0,cs-1), np.clip(int(r['center_y']*(cs-1)),0,cs-1)
        hm_all[cy,cx] += 1
    im = axes[-1].imshow(hm_all, cmap='inferno', interpolation='gaussian')
    axes[-1].set_title('COMBINED', fontweight='bold')
    plt.colorbar(im, ax=axes[-1], fraction=0.046)
    plt.tight_layout(); plt.show()

### 6.3 Bounding Box Geometry

In [ ]:
if len(df) > 0:
    pal = {'tumor':'#e74c3c','tuberculosis':'#3498db'}
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle('BBox Geometry', fontsize=14, fontweight='bold')
    for l in CLASS_NAMES:
        s=df[df['label']==l]
        axes[0].scatter(s['aspect_ratio'],s['area'],alpha=0.3,label=l,color=pal[l],s=20)
    axes[0].set_xlabel('Aspect Ratio'); axes[0].set_ylabel('Area'); axes[0].set_yscale('log')
    axes[0].set_title('Aspect Ratio vs Area',fontweight='bold'); axes[0].legend()
    for l in CLASS_NAMES:
        s=df[df['label']==l]
        axes[1].scatter(s['width'],s['height'],alpha=0.3,label=l,color=pal[l],s=20)
    axes[1].set_title('Width vs Height',fontweight='bold'); axes[1].legend()
    for l in CLASS_NAMES:
        s=df[df['label']==l]
        axes[2].hist(s['relative_area_pct'],bins=40,alpha=0.6,label=l,color=pal[l],density=True)
    axes[2].set_title('Relative Lesion Size',fontweight='bold'); axes[2].legend()
    plt.tight_layout(); plt.show()
    for l in CLASS_NAMES:
        s=df[df['label']==l]
        print(f"  {l.upper()}: W={s['width'].mean():.0f}\u00b1{s['width'].std():.0f}, H={s['height'].mean():.0f}\u00b1{s['height'].std():.0f}")

### 6.4 Annotation Density & Class Balance

In [ ]:
if len(df) > 0:
    pal = {'tumor':'#e74c3c','tuberculosis':'#3498db'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for l in CLASS_NAMES:
        bpi = df[df['label']==l].groupby('image').size()
        axes[0].hist(bpi, bins=range(1,bpi.max()+2), alpha=0.6, label=f'{l} (\u03bc={bpi.mean():.1f})', edgecolor='white')
    axes[0].set_title('BBoxes per Image',fontweight='bold'); axes[0].legend()
    cc = df['label'].value_counts()
    bars = axes[1].bar(cc.index, cc.values, color=[pal.get(l,'gray') for l in cc.index])
    for b,v in zip(bars,cc.values): axes[1].text(b.get_x()+b.get_width()/2,v+5,str(v),ha='center',fontweight='bold')
    if len(cc)>1: axes[1].set_xlabel(f'Imbalance: {cc.max()/cc.min():.1f}:1')
    axes[1].set_title('Class Distribution',fontweight='bold')
    plt.tight_layout(); plt.show()

### 6.5 Sample Grid

In [ ]:
if len(df) > 0:
    ns = 4
    fig, axes = plt.subplots(len(CLASS_NAMES), ns, figsize=(5*ns, 5*len(CLASS_NAMES)))
    fig.suptitle('Sample Images with Bounding Boxes', fontsize=16, fontweight='bold', y=1.02)
    bcol = {'tumor':(255,0,0),'tuberculosis':(0,100,255)}
    for ri,l in enumerate(CLASS_NAMES):
        samps = df[df['label']==l].drop_duplicates('image').sample(min(ns,df[df['label']==l]['image'].nunique()),random_state=seed)
        for ci,(_,sr) in enumerate(samps.iterrows()):
            img = cv2.imread(sr['image_path'])
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            for _,an in df[df['image']==sr['image']].iterrows():
                col = bcol.get(an['label'],(0,255,0))
                cv2.rectangle(img,(int(an['xmin']),int(an['ymin'])),(int(an['xmax']),int(an['ymax'])),col,3)
                cv2.putText(img,an['label'],(int(an['xmin']),int(an['ymin'])-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,col,2)
            ax = axes[ri][ci] if len(CLASS_NAMES)>1 else axes[ci]
            ax.imshow(img); ax.axis('off'); ax.set_title(f"{l}: {sr['image']}",fontsize=9)
    plt.tight_layout(); plt.show()

### 6.6 CLAHE Enhancement Comparison

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle('CLAHE Before vs After', fontsize=14, fontweight='bold')
    samps = df.drop_duplicates('image').sample(4, random_state=seed)
    clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    for i,(_,r) in enumerate(samps.iterrows()):
        img = cv2.imread(r['image_path'])
        if img is None: continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        lab[:,:,0] = clahe_obj.apply(lab[:,:,0])
        enh = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        axes[0][i].imshow(rgb); axes[0][i].set_title('Original'); axes[0][i].axis('off')
        axes[1][i].imshow(enh); axes[1][i].set_title('CLAHE'); axes[1][i].axis('off')
    plt.tight_layout(); plt.show()

## 7. Bias Analysis & Detection

### 7.1 Cross-Dataset Domain Shift

In [ ]:
if len(df) > 0 and len(quality_df) > 0:
    print("CROSS-DATASET DOMAIN SHIFT REPORT")
    print("=" * 60)
    for src in quality_df['source_dataset'].unique():
        sub = quality_df[quality_df['source_dataset']==src]
        print(f"\n  {src}:")
        for m in ['brightness','contrast','sharpness','width','height']:
            print(f"    {m:12s}: {sub[m].mean():.1f} \u00b1 {sub[m].std():.1f}")
    print("\n  Risk Assessment:")
    for m in ['brightness','contrast']:
        vals = [quality_df[quality_df['source_dataset']==s][m].mean() for s in quality_df['source_dataset'].unique()]
        if len(vals)==2:
            dp = abs(vals[0]-vals[1])/max(vals)*100
            print(f"    {m:12s}: {dp:.1f}% diff \u2192 {'HIGH' if dp>20 else 'MODERATE' if dp>10 else 'LOW'} risk")

### 7.2 Spatial & Size Bias

In [ ]:
if len(df) > 0:
    print("SPATIAL BIAS \u2014 Quadrant Distribution")
    print("=" * 55)
    for l in CLASS_NAMES:
        sub = df[df['label']==l]
        qs = [('Top' if r['center_y']<0.5 else 'Bottom')+'-'+('Left' if r['center_x']<0.5 else 'Right') for _,r in sub.iterrows()]
        ct = Counter(qs); tot = sum(ct.values())
        print(f"\n  {l.upper()}:")
        for q in ['Top-Left','Top-Right','Bottom-Left','Bottom-Right']:
            c = ct.get(q,0); print(f"    {q:15s}: {c:5d} ({c/tot*100:.1f}%)")
        obs = [ct.get(q,0) for q in ['Top-Left','Top-Right','Bottom-Left','Bottom-Right']]
        chi2,p = stats.chisquare(obs,[tot/4]*4)
        print(f"    \u03c7\u00b2={chi2:.2f}, p={p:.4f} {'\u26a0 NON-UNIFORM' if p<0.05 else '\u2713 Uniform'}")
    
    print("\n\nSIZE BIAS")
    print("=" * 55)
    for l in CLASS_NAMES:
        sub = df[df['label']==l]
        q33,q66 = sub['area'].quantile(0.33),sub['area'].quantile(0.66)
        sm,md2,lg = len(sub[sub['area']<=q33]),len(sub[(sub['area']>q33)&(sub['area']<=q66)]),len(sub[sub['area']>q66])
        tot = len(sub)
        print(f"\n  {l.upper()}: Small={sm} ({sm/tot*100:.0f}%), Med={md2} ({md2/tot*100:.0f}%), Large={lg} ({lg/tot*100:.0f}%)")

### 7.3 Bias Summary

In [ ]:
if len(df) > 0:
    from IPython.display import display, Markdown
    display(Markdown('''
### Identified Biases & Mitigations

| Bias Type | Finding | Risk | Mitigation |
|-----------|---------|------|------------|
| **Class Imbalance** | Unequal tumor vs TB | MODERATE | Weighted sampling |
| **Domain Shift** | Different imaging protocols | HIGH | CLAHE normalization |
| **Spatial Bias** | Location clustering | MODERATE | Flip & rotation augmentation |
| **Size Bias** | Small lesion underrepresentation | MODERATE | Size-aware augmentation |
| **Annotation Heterogeneity** | Masks vs COCO JSON | LOW | Unified pipeline |
'''))

## 8. Bias Mitigation

### 8.1 Weighted Random Sampling

In [ ]:
if len(df) > 0:
    ccounts = df['label'].value_counts().to_dict()
    total = sum(ccounts.values())
    cweights = {l: total/(len(ccounts)*c) for l,c in ccounts.items()}
    print("Class Weights:", {l:round(w,3) for l,w in cweights.items()})
    
    img_labels = df.groupby('image')['label'].first()
    image_weights = {img: cweights[l] for img,l in img_labels.items()}
    
    # Size-aware boost
    img_max_area = df.groupby('image')['area'].max()
    area_med = img_max_area.median()
    boosted = 0
    for img in image_weights:
        if img in img_max_area.index and img_max_area[img] < area_med * 0.5:
            image_weights[img] *= 1.5; boosted += 1
    print(f"Size boost applied to {boosted} small-lesion images")

### 8.2 Mitigation Preview

In [ ]:
if len(df) > 0:
    pal = {'tumor':'#e74c3c','tuberculosis':'#3498db'}
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Before vs After Weighted Sampling', fontsize=14, fontweight='bold')
    cc = img_labels.value_counts()
    axes[0].bar(cc.index, cc.values, color=[pal.get(l,'gray') for l in cc.index])
    axes[0].set_title('Before', fontweight='bold')
    
    all_imgs = list(image_weights.keys())
    wl = [image_weights[i] for i in all_imgs]
    ll = [img_labels[i] for i in all_imgs]
    probs = np.array(wl)/sum(wl)
    si = np.random.choice(len(all_imgs), size=min(2000,len(all_imgs)*3), replace=True, p=probs)
    sc = Counter([ll[i] for i in si])
    axes[1].bar(sc.keys(), sc.values(), color=[pal.get(l,'gray') for l in sc.keys()])
    axes[1].set_title('After (simulated)', fontweight='bold')
    plt.tight_layout(); plt.show()

## 9. Image Preprocessing Pipeline

### 9.1 CLAHE & Bilateral Denoising

In [ ]:
def preprocess_image(img_path, apply_clahe=True, apply_denoise=True):
    img = cv2.imread(img_path)
    if img is None: return None
    if apply_clahe:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        lab[:,:,0] = cl.apply(lab[:,:,0])
        img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    if apply_denoise:
        img = cv2.bilateralFilter(img, d=5, sigmaColor=50, sigmaSpace=50)
    return img

if len(df) > 0:
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    fig.suptitle('Preprocessing Stages', fontsize=14, fontweight='bold')
    samps = df.drop_duplicates('image').sample(4, random_state=seed)
    for i,(_,r) in enumerate(samps.iterrows()):
        o = cv2.cvtColor(cv2.imread(r['image_path']),cv2.COLOR_BGR2RGB)
        c = cv2.cvtColor(preprocess_image(r['image_path'],True,False),cv2.COLOR_BGR2RGB)
        f = cv2.cvtColor(preprocess_image(r['image_path']),cv2.COLOR_BGR2RGB)
        axes[0][i].imshow(o); axes[0][i].set_title('Original'); axes[0][i].axis('off')
        axes[1][i].imshow(c); axes[1][i].set_title('+CLAHE'); axes[1][i].axis('off')
        axes[2][i].imshow(f); axes[2][i].set_title('+CLAHE+Denoise'); axes[2][i].axis('off')
    plt.tight_layout(); plt.show()

### 9.2 Normalization Statistics

In [ ]:
if len(df) > 0:
    print("Computing channel statistics...")
    csums, csq, px = np.zeros(3), np.zeros(3), 0
    sp = random.sample(df.drop_duplicates('image')['image_path'].tolist(), min(500,df['image'].nunique()))
    for p in tqdm(sp, desc="Stats"):
        img = cv2.imread(p)
        if img is None: continue
        img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB).astype(np.float64)/255.0
        csums += img.sum(axis=(0,1)); csq += (img**2).sum(axis=(0,1))
        px += img.shape[0]*img.shape[1]
    dm, ds = csums/px, np.sqrt(csq/px-(csums/px)**2)
    print(f"\nDataset: Mean=[{dm[0]:.4f},{dm[1]:.4f},{dm[2]:.4f}], Std=[{ds[0]:.4f},{ds[1]:.4f},{ds[2]:.4f}]")
    print(f"ImageNet: Mean=[0.485,0.456,0.406], Std=[0.229,0.224,0.225]")
    print("\u2192 Using ImageNet normalization for pretrained YOLOX compatibility")
    norm_stats = ([0.485,0.456,0.406],[0.229,0.224,0.225])

## 10. Stratified Splitting with Leakage Prevention

In [ ]:
if len(df) > 0:
    il = df.groupby('image').agg({'label':'first','image_path':'first'}).reset_index()
    tr, tmp = train_test_split(il, test_size=0.20, random_state=seed, stratify=il['label'])
    vl, ts = train_test_split(tmp, test_size=0.50, random_state=seed, stratify=tmp['label'])
    train_keys, val_keys, test_keys = tr['image'].tolist(), vl['image'].tolist(), ts['image'].tolist()
    
    assert not (set(train_keys)&set(val_keys)) and not (set(train_keys)&set(test_keys))
    
    if 'hash_df' in dir() and len(hash_df)>0:
        th = set(hash_df[hash_df['image'].isin(train_keys)]['phash'])
        tth = set(hash_df[hash_df['image'].isin(test_keys)]['phash'])
        ho = th & tth
        print(f"{'\u26a0 '+str(len(ho))+' hash overlaps' if ho else '\u2713 No hash leakage'}")
    
    print(f"\nSplit: Train={len(train_keys)}, Val={len(val_keys)}, Test={len(test_keys)}")
    for sn,sk in [('Train',train_keys),('Val',val_keys),('Test',test_keys)]:
        sd = df[df['image'].isin(sk)]
        parts = [f"{l}: {sd[sd['label']==l]['image'].nunique()} imgs" for l in CLASS_NAMES]
        print(f"  {sn:6s}: {' | '.join(parts)}")
    
    tsw = [image_weights.get(i,1.0) for i in train_keys]
    sampler = WeightedRandomSampler(tsw, num_samples=len(train_keys), replacement=True)
    print("\u2713 WeightedRandomSampler configured")

## 11. Custom Dataset

In [ ]:
class LungDiseaseDataset(Dataset):
    def __init__(self, img_keys, df, class_to_idx, transforms=None, apply_clahe=True):
        self.img_keys = img_keys
        self.df = df.set_index('image')
        self.class_to_idx = class_to_idx
        self.transforms = transforms
        self.apply_clahe = apply_clahe
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    def __len__(self): return len(self.img_keys)
    
    def __getitem__(self, idx):
        nm = self.img_keys[idx]
        ip = self.df.loc[nm, 'image_path']
        if isinstance(ip, pd.Series): ip = ip.iloc[0]
        ic = cv2.imread(str(ip))
        if ic is None: ic = np.zeros((512,512,3), dtype=np.uint8)
        if self.apply_clahe:
            lab = cv2.cvtColor(ic, cv2.COLOR_BGR2LAB)
            lab[:,:,0] = self.clahe.apply(lab[:,:,0])
            ic = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
        ic = cv2.cvtColor(ic, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(ic)
        rows = self.df.loc[[nm]]
        bboxes = torch.tensor(rows[['xmin','ymin','xmax','ymax']].values.astype(float), dtype=torch.float32)
        labels = torch.tensor([self.class_to_idx[l] for l in rows['label'].values], dtype=torch.long)
        target = {'boxes': BoundingBoxes(bboxes, format="xyxy", canvas_size=image.size[::-1]), 'labels': labels}
        if self.transforms: image, target = self.transforms(image, target)
        return image, target

## 12. Advanced Augmentation Pipeline

In [ ]:
fill = (0,0,0)
norm_stats = ([0.485,0.456,0.406],[0.229,0.224,0.225])

resize_pad_tfm = transforms.Compose([
    ResizeMax(max_sz=train_sz), PadSquare(shift=True, fill=fill),
    transforms.Resize([train_sz]*2, antialias=True)
])
final_tfms = transforms.Compose([
    transforms.ToImage(), transforms.ToDtype(torch.float32, scale=True),
    transforms.SanitizeBoundingBoxes(), transforms.Normalize(*norm_stats),
])
train_tfms = transforms.Compose([
    CustomRandomAugment(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15, fill=fill),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05,0.05), scale=(0.9,1.1), fill=fill),
    resize_pad_tfm, final_tfms,
])
valid_tfms = transforms.Compose([resize_pad_tfm, final_tfms])

if len(df) > 0:
    train_dataset = LungDiseaseDataset(train_keys, df, CLASS_TO_IDX, train_tfms, apply_clahe=True)
    valid_dataset = LungDiseaseDataset(val_keys, df, CLASS_TO_IDX, valid_tfms, apply_clahe=True)
    dlp = {'batch_size':bs,'num_workers':0,'collate_fn':lambda b:tuple(zip(*b)),
           'pin_memory':'cuda' in device,'pin_memory_device':device if 'cuda' in device else ''}
    train_loader = DataLoader(train_dataset, **dlp, sampler=sampler, drop_last=True)
    valid_loader = DataLoader(valid_dataset, **dlp, drop_last=False)
    print(f"Train: {len(train_loader)} batches, Val: {len(valid_loader)} batches")

### 12.1 Augmentation Showcase

In [ ]:
if len(df) > 0:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    fig.suptitle('Same Image \u00d7 8 Random Augmentations', fontsize=14, fontweight='bold')
    axes = axes.flatten()
    colors = distinctipy.get_colors(NUM_CLASSES)
    int_colors = [tuple(int(c*255) for c in col) for col in colors]
    for i in range(8):
        ai, at = train_dataset[0]
        it = denorm_img_tensor(ai, norm_stats)
        iu = transforms.ToDtype(torch.uint8, scale=True)(it)
        lbs = [CLASS_NAMES[int(l)] for l in at['labels']]
        if len(at['boxes'])>0:
            ann = draw_bounding_boxes(image=iu,boxes=at['boxes'],labels=lbs,
                colors=[int_colors[CLASS_NAMES.index(l)] for l in lbs],width=2)
        else: ann = iu
        axes[i].imshow(tensor_to_pil(ann)); axes[i].set_title(f'Aug #{i+1}'); axes[i].axis('off')
    plt.tight_layout(); plt.show()

## 13. Data Integrity Validation

In [ ]:
if len(df) > 0:
    print("=" * 60)
    print("FINAL DATA INTEGRITY VALIDATION")
    print("=" * 60)
    ok = 0
    # 1. No leakage
    if not (set(train_keys)&set(val_keys)) and not (set(train_keys)&set(test_keys)) and not (set(val_keys)&set(test_keys)):
        print("\u2713 [1/5] No data leakage"); ok+=1
    # 2. Images load
    errs = sum(1 for i in random.sample(train_keys,min(50,len(train_keys))) if cv2.imread(df[df['image']==i]['image_path'].iloc[0]) is None)
    if errs==0: print("\u2713 [2/5] All images load"); ok+=1
    else: print(f"\u2717 [2/5] {errs} load failures")
    # 3. BBox bounds
    be = 0
    for _,r in df.sample(min(200,len(df))).iterrows():
        img = cv2.imread(r['image_path'])
        if img is not None:
            h,w = img.shape[:2]
            if r['xmin']<0 or r['ymin']<0 or r['xmax']>w or r['ymax']>h: be+=1
    if be==0: print("\u2713 [3/5] BBoxes within bounds"); ok+=1
    else: print(f"\u2717 [3/5] {be} OOB bboxes")
    # 4. DataLoader
    try:
        for bi,bt in train_loader: assert len(bi)==bs; break
        print("\u2713 [4/5] DataLoader OK"); ok+=1
    except Exception as e: print(f"\u2717 [4/5] {e}")
    # 5. Class coverage
    tl = df[df['image'].isin(train_keys)]['label'].unique()
    if set(tl)==set(CLASS_NAMES): print("\u2713 [5/5] All classes in train"); ok+=1
    
    print(f"\n{'='*60}")
    print(f"RESULT: {ok}/5 passed {'\u2713 READY' if ok==5 else ''}")
    
    from IPython.display import display, Markdown
    display(Markdown(f'''
### Preprocessing Summary
| Metric | Value |
|--------|-------|
| Total Images | {df["image"].nunique()} |
| Total Annotations | {len(df)} |
| Classes | {", ".join(CLASS_NAMES)} |
| Train / Val / Test | {len(train_keys)} / {len(val_keys)} / {len(test_keys)} |
| Resolution | {train_sz}\u00d7{train_sz} |
| Preprocessing | CLAHE + Bilateral Filter |
| Normalization | ImageNet |
| Augmentation | Flip, Rotate, ColorJitter, Affine, CustomRandomAugment |
| Bias Mitigation | Weighted sampling + size-aware boost |
'''))

## 14. Model & Training

In [ ]:
if len(df) > 0:
    model_type = 'yolox_tiny'
    model = build_model(model_type, NUM_CLASSES, pretrained=True).to(device)
    loss_func = YOLOXLoss(num_classes=NUM_CLASSES, bbox_loss_weight=10.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs*len(train_loader))
    print(f"Model: {model_type} ({NUM_CLASSES} classes: {CLASS_NAMES})")

In [ ]:
def run_epoch(model, loader, optimizer, scheduler, loss_func, device, scaler, is_train):
    model.train() if is_train else model.eval()
    total_loss = 0
    if len(loader)==0: return 0.0
    pbar = tqdm(loader, desc="Train" if is_train else "Eval")
    for bid,(inputs,targets) in enumerate(pbar):
        inputs = torch.stack(inputs).to(device)
        gb = [t['boxes'].to(device) for t in targets]
        gl = [t['labels'].to(device) for t in targets]
        with autocast(device_type=torch.device(device).type):
            cs,bp,os2 = model(inputs)
            losses = loss_func(cs,bp,os2,gb,gl)
            loss = sum(losses.values())
        if is_train:
            optimizer.zero_grad()
            if scaler: scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else: loss.backward(); optimizer.step()
            scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item(), avg=total_loss/(bid+1))
        if is_train and not math.isfinite(loss.item()): print(f"Loss={loss.item()}, stopping"); break
    return total_loss/len(loader)

if len(df) > 0:
    scaler = GradScaler() if 'cuda' in device else None
    best_val_loss = float('inf')
    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
        tl = run_epoch(model, train_loader, optimizer, scheduler, loss_func, device, scaler, True)
        with torch.no_grad():
            vl = run_epoch(model, valid_loader, None, None, loss_func, device, scaler, False)
        if vl < best_val_loss and vl > 0:
            best_val_loss = vl
            torch.save(model.state_dict(), project_dir/"best_model.pth")
            print("Saved best model!")

## 15. Inference & Test Visualization

In [ ]:
if len(df) > 0 and (project_dir/"best_model.pth").exists():
    model.load_state_dict(torch.load(project_dir/"best_model.pth"))
    model.eval()
    mean_t = torch.tensor(norm_stats[0]).view(1,3,1,1).to(device)
    std_t = torch.tensor(norm_stats[1]).view(1,3,1,1).to(device)
    wrapped_model = YOLOXInferenceWrapper(model, mean_t, std_t)
    
    def predict(img_path):
        img = Image.open(img_path).convert("RGB")
        resized = resize_img(img, target_sz=train_sz)
        inp = transforms.Compose([transforms.ToImage(),transforms.ToDtype(torch.float32,scale=True)])(resized)[None].to(device)
        with torch.no_grad(): out = wrapped_model(inp).cpu()
        mask = out[0,:,-1] > 0.3
        props = out[0,mask]
        if len(props)==0: return resized
        boxes = torchvision.ops.box_convert(props[:,:4],'xywh','xyxy')
        labs = [CLASS_NAMES[int(i)] for i in props[:,4]]
        cols = distinctipy.get_colors(NUM_CLASSES)
        ic = [tuple(int(c*255) for c in col) for col in cols]
        return tensor_to_pil(draw_bounding_boxes(
            image=transforms.PILToTensor()(resized), boxes=boxes, labels=labs,
            colors=[ic[CLASS_NAMES.index(l)] for l in labs], width=2))

### Test Set Predictions

In [ ]:
if len(df) > 0 and (project_dir/"best_model.pth").exists():
    print(f"Best Val Loss: {best_val_loss:.4f}")
    nt = min(6, len(test_keys))
    if nt > 0:
        fig, axes = plt.subplots(nt, 2, figsize=(12, 5*nt))
        if nt==1: axes=[axes]
        for i in range(nt):
            tn = test_keys[i]
            tp = df[df['image']==tn]['image_path'].iloc[0]
            axes[i][0].imshow(Image.open(tp).convert("RGB")); axes[i][0].set_title(f"Original: {tn}"); axes[i][0].axis('off')
            axes[i][1].imshow(predict(tp)); axes[i][1].set_title("Prediction"); axes[i][1].axis('off')
        plt.tight_layout(); plt.show()